# 04 — Demo
Upload any face-centered photo and get a mask classification prediction.

**Requires:** `face_mask_classifier.pth` from notebook 02.

**Note:** Input should be a face-centered crop. Wide shots with significant background may reduce accuracy — this is a known limitation documented in the project README.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

CLASS_NAMES = ['mask_weared_incorrect', 'with_mask', 'without_mask']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Upload face_mask_classifier.pth:')
files.upload()

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 3)
model.load_state_dict(torch.load('face_mask_classifier.pth', map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print('Model ready.')

## Run Inference
Upload a face photo below. Re-run this cell for each new image.

In [ ]:
# Threshold: if mask_weared_incorrect confidence exceeds this while
# top prediction is with_mask, override the prediction.
# Required due to dataset underrepresentation of nose-exposed incorrect wearing.
INCORRECT_THRESHOLD = 0.20

print('Upload an image to classify:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

img = Image.open(img_path).convert('RGB')
tensor = eval_transforms(img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model(tensor)
    probs  = torch.softmax(output, dim=1)[0]

pred = probs.argmax().item()
predicted_class = CLASS_NAMES[pred]

if probs[0].item() > INCORRECT_THRESHOLD and predicted_class == 'with_mask':
    predicted_class = 'mask_weared_incorrect'

print(f'Prediction: {predicted_class}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {probs[i].item():.2%}')

plt.imshow(img)
plt.title(f'Predicted: {predicted_class}')
plt.axis('off')
plt.show()